<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. 데이터 불러오기
try:
    df = pd.read_spss("/content/HN24_ALL.sav")
except ImportError:
    import os
    os.system('pip install pyreadstat')
    df = pd.read_spss("/content/HN24_ALL.sav")

# 변수명 매핑
sex_col = 'sex'
age_col = 'age'
dm_col = 'HE_DM_HbA1c'
hp_col = 'HE_HP'
ob_col = 'HE_obe'
incm_col = 'incm'
edu_col = 'edu'
bs_col = 'BS3_1'
bd_col = 'BD1_11'

# 빈도표 출력용 전체 변수 리스트
check_vars = [sex_col, age_col, dm_col, hp_col, ob_col, incm_col, edu_col, bs_col, bd_col]

# ---------------------------------------------------------
# [사전 확인] 주요 변수 원본 빈도표 (결측 및 8, 9 제외 안 함)
# ---------------------------------------------------------
print("=== [사전 확인] 주요 변수 원본 빈도표 (모든 코드 및 결측 포함) ===")
for var in check_vars:
    print(f"\n▶ 변수명: [{var}]")
    temp_series = pd.to_numeric(df[var], errors='coerce')

    # 별도의 필터링 없이 value_counts()로 전체 분포 확인 (결측치는 dropna=False)
    freq_table = temp_series.value_counts(dropna=False).sort_index()
    freq_df = pd.DataFrame({
        '코드(응답/결측)': [str(int(x)) if pd.notnull(x) else '결측(NaN)' for x in freq_table.index],
        '빈도수(N)': freq_table.values
    })
    display(freq_df)

# ---------------------------------------------------------
# [데이터 정제] 분석 대상자 선정 (Flow Chart)
# - 노출/결과 변수까지만 결측·8·9를 제외하고, 공변량은 제외하지 않음
# ---------------------------------------------------------
print("\n=== [데이터 정제] 분석 대상자 선정 단계 ===")
print(f"Step 0. 2024년 KNHANES 원본 데이터: N = {len(df):,}")

# 2. 포함 기준 (만 19세 이상)
df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
df_step1 = df[df[age_col] >= 19].copy()
print(f"Step 1. 만 19세 이상 성인 포함: N = {len(df_step1):,} (제외: {len(df) - len(df_step1):,}명)")

# 3. 노출변수(당뇨병) 결측(.), 8(비해당), 9(모름) 제외
df_step1[dm_col] = pd.to_numeric(df_step1[dm_col], errors='coerce')
df_step2 = df_step1[df_step1[dm_col].notnull() & (~df_step1[dm_col].isin([8, 9]))].copy()
print(f"Step 2. 노출변수({dm_col}) 결측/8/9 제외: N = {len(df_step2):,} (제외: {len(df_step1) - len(df_step2):,}명)")

# 4. 결과변수(고혈압) 결측(.), 8(비해당), 9(모름) 제외
df_step2[hp_col] = pd.to_numeric(df_step2[hp_col], errors='coerce')
df_step3 = df_step2[df_step2[hp_col].notnull() & (~df_step2[hp_col].isin([8, 9]))].copy()
print(f"Step 3. 결과변수({hp_col}) 결측/8/9 제외: N = {len(df_step3):,} (제외: {len(df_step2) - len(df_step3):,}명)")

# 5. 공변량에 대한 강제 제외(Drop) 과정을 수행하지 않고 최종 기본 데이터셋 확정
df_final = df_step3.copy()

print("-" * 50)
print(f">> 최종 기본 분석 대상자 수 (공변량 미제외): N = {len(df_final):,}명")
print("※ 공변량들의 결측치, 8(비해당), 9(모름)는 대상자 선정 시 제외되지 않고 그대로 유지되었습니다.")

=== [사전 확인] 주요 변수 원본 빈도표 (모든 코드 및 결측 포함) ===

▶ 변수명: [sex]


,코드(응답/결측),빈도수(N)
0,1,3070
1,2,3927



▶ 변수명: [age]


,코드(응답/결측),빈도수(N)
0,1,30
1,2,30
2,3,40
3,4,38
4,5,37
...,...,...
75,76,105
76,77,81
77,78,55
78,79,55



▶ 변수명: [HE_DM_HbA1c]


,코드(응답/결측),빈도수(N)
0,1,2813
1,2,1844
2,3,959
3,결측(NaN),1381


... (9 outputs left out)

,코드(응답/결측),빈도수(N)
0,1,681
1,2,118
2,3,1368
3,8,4605
4,9,216
5,결측(NaN),9



▶ 변수명: [BD1_11]


,코드(응답/결측),빈도수(N)
0,1,1115
1,2,1179
2,3,671
3,4,1219
4,5,800
5,6,292
6,8,1496
7,9,216
8,결측(NaN),9



=== [데이터 정제] 분석 대상자 선정 단계 ===
Step 0. 2024년 KNHANES 원본 데이터: N = 6,997
Step 1. 만 19세 이상 성인 포함: N = 6,033 (제외: 964명)
Step 2. 노출변수(HE_DM_HbA1c) 결측/8/9 제외: N = 5,616 (제외: 417명)
Step 3. 결과변수(HE_HP) 결측/8/9 제외: N = 5,567 (제외: 49명)
--------------------------------------------------
>> 최종 기본 분석 대상자 수 (공변량 미제외): N = 5,567명
※ 공변량들의 결측치, 8(비해당), 9(모름)는 대상자 선정 시 제외되지 않고 그대로 유지되었습니다.


In [36]:
# ---------------------------------------------------------
# [사전 분석] 변수 및 복합표본설계 변수, 원 코드 빈도 확인
# ---------------------------------------------------------

# 1. 복합표본설계 변수 (층, 집락, 가중치) 확인
# SPSS 원시자료의 대소문자 차이를 방지하기 위해 일치하는 컬럼을 찾습니다.
target_design_vars = ['kstrata', 'psu', 'wt_itvex']
design_cols = [col for col in df.columns if col.lower() in target_design_vars]

print("=== 1. 복합표본설계 변수 확인 ===")
if len(design_cols) == 3:
    print(f"분석에 필요한 설계 변수가 모두 존재합니다: {design_cols}")
    # 가중치의 요약 통계량(최소, 최대, 평균 등)을 확인하여 이상치 여부 점검
    display(df[design_cols].describe())
else:
    print(f"경고: 일부 설계 변수를 찾을 수 없습니다. 현재 확인된 변수: {design_cols}")

# 2. 분석 주요 변수 리스트 (앞서 정의한 매핑 변수 사용)
check_vars = [age_col, dm_col, hp_col, ob_col, incm_col, edu_col, bs_col, bd_col]

=== 1. 복합표본설계 변수 확인 ===
분석에 필요한 설계 변수가 모두 존재합니다: ['psu', 'wt_itvex', 'kstrata']


,wt_itvex,kstrata
count,6997.000000,6997.000000
mean,7363.446477,111.520509
std,3807.130476,7.353178
min,314.853524,101.000000
25%,4738.510601,106.000000
50%,6665.953013,110.000000
75%,9024.211026,117.000000
max,30575.037524,127.000000
